# Day 39: Prompt Caching

Every chat turn re-sends the *entire* system prompt and history — and you pay full price for it, again and again.

Today we let Gemini **reuse** the repeated part. Same tokens, ~10% of the cost. We'll watch the savings live and prove it in **LangSmith**.

## Install

In [ ]:
%pip install langchain-google-genai --quiet

## Setup + LangSmith

Two extra env vars turn on **LangSmith** tracing — now every call is logged with its token usage.

In [29]:
import os
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI

load_dotenv(dotenv_path='../.env')
os.environ["GOOGLE_API_KEY"] = os.environ["GEMINI_API_KEY2"].strip()
os.environ["LANGSMITH_TRACING"] = "true"  # traces show up at smith.langchain.com

model = ChatGoogleGenerativeAI(model="gemini-2.5-flash-lite")
print("✅ Model ready, tracing on")

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


✅ Model ready, tracing on


## The Repeated Part

Real apps send a big, unchanging chunk on every call — a system prompt, a manual, a knowledge base. Here's ours.

In [30]:
import uuid

# A fresh marker each session → the first call is always a cold miss (nothing cached yet)
session = uuid.uuid4().hex[:8]

knowledge_base = (
    f"[session {session}] You are a support agent for Acme Corp. Answer only from this knowledge base.\n"
    + ("Acme was founded in 1998 in Austin, Texas. Employees get 20 vacation days. "
       "Returns are accepted within 30 days. Support hours are 9 to 5 Eastern. " * 350)
)
print(f"Knowledge base ≈ {len(knowledge_base.split())} words — sent on every question")

Knowledge base ≈ 9116 words — sent on every question


## First Call — Nothing Cached Yet

The whole knowledge base is processed fresh, at full price.

In [31]:
def ask(question):
    msg = model.invoke(knowledge_base + "\n\nQuestion: " + question)
    u = msg.usage_metadata
    cached = (u.get("input_token_details") or {}).get("cache_read", 0)
    print(f"input tokens: {u['input_tokens']:>6}   from cache: {cached:>6}")
    return msg

ask("When was Acme founded?");

input tokens:  14390   from cache:      0


## Ask Again — Cache Kicks In

Gemini's **implicit caching** is free and automatic. It just needs the same prefix, a beat to register, so we ask until it lands.

In [32]:
import time

cached = 0

msg = ask("What is the return policy?")
cached = (msg.usage_metadata.get("input_token_details") or {}).get("cache_read", 0)

print(f"\n🎯 {cached} tokens served from cache")

input tokens:  14391   from cache:  14328

🎯 14328 tokens served from cache


## The Savings

Cached tokens bill at **~10%** of normal. Same answer, fraction of the cost.

In [33]:
u = msg.usage_metadata
total = u["input_tokens"]
cached = (u.get("input_token_details") or {}).get("cache_read", 0)
fresh = total - cached

cost_no_cache = total                 # everything full price
cost_cached = fresh + cached * 0.10    # cached part at 10%
savings = (1 - cost_cached / cost_no_cache) * 100

print(f"Without caching: {cost_no_cache:.0f} token-units")
print(f"With caching:    {cost_cached:.0f} token-units")
print(f"💰 {savings:.0f}% cheaper — for the exact same request")

Without caching: 14391 token-units
With caching:    1496 token-units
💰 90% cheaper — for the exact same request


## See It in LangSmith

Open [smith.langchain.com](https://smith.langchain.com) → your project. Each call is traced with its token usage — the second one shows tokens read from cache instead of processed fresh.

## Key Takeaways

1. **You re-send the same prompt every turn** — caching turns that repetition into a ~90% discount
2. **Gemini's implicit caching is free and automatic** — same prefix, and `usage_metadata` reports `cache_read`
3. **LangSmith makes it visible** — trace every call and watch the cached-token count climb

---

Next: Day 40 — How Prompt Caching Actually Works